# 04 — Aggregate calendar weeks and inspect representation
Primary target: median of recorded daily prices for each item–state–district–premise-type
series in a Monday–Sunday week. It is an observed monitoring statistic, not a fixed-shop
price index. Frequent monitoring gives a premise more weight.

Also compute equal-weighted premise medians and same-premise consecutive-week changes.
These **future-dependent diagnostics must never become prediction inputs**.
Remove incomplete boundary weeks; do not assume seven monitoring days are required.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Run from the repository root or its notebooks folder.")
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import numpy as np
import pricecatcher_config as cfg
from data_utils import save_json, surge_event
from data_utils import aggregate_weeks
output = cfg.stage(4)
daily = pd.read_parquet(cfg.stage(3) / "daily.parquet")
daily["date"] = pd.to_datetime(daily["date"])
weekly = aggregate_weeks(daily)
weekly["week_end"] = weekly["week_start"] + pd.Timedelta(days=6)
start = pd.Period(cfg.START_MONTH).start_time.normalize()
end = pd.Period(cfg.END_MONTH).end_time.normalize()
complete = weekly.loc[weekly["week_start"].ge(start) & weekly["week_end"].le(end)].copy()
if complete.empty or complete.duplicated(cfg.SERIES_KEYS + ["week_start"]).any():
    raise ValueError("No complete weeks or duplicate weekly keys.")
complete = complete.sort_values(cfg.SERIES_KEYS + ["week_start"])
complete.to_parquet(output / "weekly.parquet", index=False)
print("Weekly rows:", len(complete), "Boundary rows removed:", len(weekly) - len(complete))


Weekly rows: 499498 Boundary rows removed: 13328


## Training-period gaps and local coverage
Use training-only statistics for methodological decisions. A remote district with few
observations should be reported as insufficient coverage, rather than shown as low risk.

In [2]:
training = complete.loc[complete["week_end"] <= pd.Timestamp(cfg.TRAIN_END)]
coverage = training.groupby(cfg.SERIES_KEYS).agg(
    weeks_present=("week_start", "size"), first_week=("week_start", "min"),
    last_week=("week_start", "max"), median_premises=("premise_count", "median"),
    median_records=("record_count", "median"))
coverage["missing_weeks_in_observed_span"] = ((coverage["last_week"] - coverage["first_week"]).dt.days // 7 + 1) - coverage["weeks_present"]
coverage.to_csv(output / "training_series_coverage.csv")
display(coverage.describe())
print("Training series with fewer than 10 observed weeks:", int(coverage["weeks_present"].lt(10).sum()))
print("Single-record training weeks:", int(training["record_count"].eq(1).sum()))


,weeks_present,first_week,last_week,median_premises,median_records,missing_weeks_in_observed_span
count,9577.000000,9577,9577,9577.000000,9577.000000,9577.000000
mean,34.369844,2025-02-18 02:50:21.488000,2025-11-18 20:10:14.973000,2.099457,8.145714,5.733319
min,1.000000,2025-01-06 00:00:00,2025-01-06 00:00:00,1.000000,1.000000,0.000000
25%,20.000000,2025-01-06 00:00:00,2025-12-15 00:00:00,1.000000,1.000000,0.000000
50%,44.000000,2025-01-06 00:00:00,2025-12-22 00:00:00,1.000000,4.000000,3.000000
75%,51.000000,2025-02-17 00:00:00,2025-12-22 00:00:00,2.000000,9.000000,7.000000
max,51.000000,2025-12-22 00:00:00,2025-12-22 00:00:00,14.000000,98.000000,48.000000
std,17.514328,NaN,NaN,1.815288,11.381114,7.716608


Training series with fewer than 10 observed weeks: 1466
Single-record training weeks: 55115
